# 09 Building the Agent Runtime

## 真正让系统像 Agent 的，不是模型本身，而是 Runtime 把推理变成闭环

前面几章其实一直在做铺垫。先解释模型为什么会表现得像在执行任务，再解释 control plane 如何塑形输出，再解释 tool calling 如何把自然语言决策压缩成结构化动作意图，再把 Agent 和 MCP 的角色分开，最后把本地模型和本地能力层都接出来。到了这一章，前面的所有东西终于要落到同一个地方：Runtime。

如果没有这一层，整套系统永远只是一些彼此看起来相关的能力：模型会回答、工具能调用、资源能读取、MCP server 能暴露能力。但这些能力本身不会自动组成任务执行链。真正把它们变成 Agent 的，是一个持续维护目标、解释模型输出、调度外部能力、吸收反馈并决定何时停止的运行时。

所以，这一章的重点不是再给 Agent 下定义，而是把 Agent Runtime 当成系统中最关键的编排层来拆。这里才是“像 Agent”与“真的有 Agent 结构”之间的分界线。

## 先给结论

这一章最重要的判断可以压缩成一句话：

> Agent Runtime 的本质，是把模型生成的下一步意图，持续解释成受目标约束的系统动作，并把动作结果重新变成下一轮推理条件。

这句话里包含了 Runtime 的几个核心职责：

- 它不是单次调用包装层，而是循环组织层
- 它不是替模型思考，而是替系统维护任务结构
- 它不是直接提供能力，而是消费并调度能力层
- 它不是只在成功路径存在，而是在失败和偏差中也要保持任务可控

可以说，前面所有讨论最终都在为这一层服务。因为只有 Runtime 把这些元素组织起来，模型、MCP、工具、资源、提示模板才不再是散点。

## 1. 为什么 Runtime 才是整个系统的中枢

很多人会下意识把模型当成系统中心，仿佛一切智能都来自模型参数本身。这种直觉在单轮问答里问题不大，但一旦系统进入 Agent 结构，它就会变得不够用。

原因很简单：模型只会在当前上下文里生成下一步最像合理输出的内容，但它不会自动承担以下这些事情：

- 当前任务是否已经偏离目标
- 现在该读 resource 还是调 tool
- 这次输出如果结构不合法该怎么办
- 连续两轮没进展是否该停止
- 某个工具失败后是否要重试还是换路径

这些问题都不是模型自然会替你处理的。Runtime 之所以是系统中枢，恰恰因为它承担的是任务秩序维护，而不是语言生成本身。

## 2. Runtime 实际上在消费哪些输入

把 Runtime 理解清楚的一个好办法，是先看它到底在消费什么。对这套项目来说，Runtime 至少同时面对四类输入：

- 当前任务目标和状态
- 模型返回的自然语言或结构化动作意图
- MCP 能力层暴露出来的 tools / resources / prompts
- 每一步动作返回的外部结果或错误信息

也就是说，Runtime 从来不是只在“模型输出之后”才开始工作。它实际上从任务一开始就在工作：准备控制信息、组织上下文、选择暴露给模型的能力面、解释模型响应、更新状态、决定是否继续。

一旦从这个角度去看，Runtime 就不再像一个小型 helper，而更像系统的调度核心。

## 3. Runtime 的第一步不是调用模型，而是定义任务入口

很多实现会把 Runtime 的起点写成“收到用户消息 -> 调模型”。这种写法对聊天程序够用，但对 Agent 系统来说太薄。

更准确的起点应该是：Runtime 收到一个任务后，先把它整理成可执行的入口状态。这个入口至少包括：

- 当前主目标
- 初始上下文材料
- 可以暴露给模型的能力面
- 本轮任务适用的 system prompt 或 prompt 模板
- 成功标准和停止条件

换句话说，Runtime 的第一步不是“让模型想一想”，而是先把模型要进入的任务环境搭起来。环境一旦搭错，后面的思考和动作都会带偏。

In [ ]:
runtime_state = {
    "goal": "基于岗位 JD 和候选人资料生成结构化能力匹配分析",
    "step": 0,
    "messages": [],
    "known_facts": [],
    "available_tools": ["extract_key_requirements", "score_candidate_fit"],
    "available_resources": ["project://case/jd_sample", "project://case/candidate_profile"],
    "termination": {"max_steps": 8},
}

for key, value in runtime_state.items():
    print(f"{key}: {value}")

## 4. Runtime 最关键的能力，是解释模型意图

模型会输出什么？在一个任务型系统里，常见情况至少有三种：

- 直接输出最终或阶段性回答
- 输出需要读取 resource 的信号
- 输出需要调用 tool 的结构化意图

Runtime 最核心的职责之一，就是把这些不同类型的输出解释成系统下一步动作。这个解释过程绝不是机械转发，而是一个小型控制决策点。

例如：

- 如果模型给出的是 tool call，但参数不完整，Runtime 不能直接盲跑
- 如果模型想直接回答，但当前 system prompt 明确要求先验证，Runtime 需要拒绝这种捷径
- 如果模型请求读取一个不存在的 resource，Runtime 需要返回明确失败信息，而不是默默吞掉

也就是说，Runtime 并不是模型的秘书，而是模型输出的解释器和守门员。

## 5. Resource Read Loop：Agent 不只是调工具，更是调上下文

很多实现一说到 Agent，就立即把注意力放在工具调用上。但对真实任务来说，先读资源再行动往往才是更常见的路径。

因此，一个像样的 Runtime 不能只实现 tool loop，还要实现 resource loop。典型流程可能是：

1. 模型判断当前信息不足
2. Runtime 允许读取某个 resource
3. Resource 内容被压缩或原样回填进上下文
4. 模型基于新上下文重新判断下一步

这条链非常重要，因为它决定系统会不会总是把问题粗暴地转成“调用工具”，而忽略其实更适合先看资料、先读文档、先理解约束。也正因为有 resource loop，MCP 的 resource 层才不会只是摆设。

## 6. Prompt Template Selection：Runtime 还要决定怎么进入任务

前面已经说过，prompt 不只是字符串，而是任务入口能力。到了 Runtime 这一层，这件事就会变得很具体：并不是所有任务都应该从同一种 system prompt 和同一种消息骨架进入。

有些任务适合直接走通用 Agent prompt；有些任务则更适合先套一个领域模板，例如：

- 候选人匹配分析
- 项目文档结构化摘要
- 需求拆解与风险识别

如果 Runtime 能根据任务类型选择合适的 prompt 模板，模型进入问题的姿态就会更稳定；如果所有任务都塞进同一段泛化提示词，系统就更容易在风格、结构和重点上漂移。

## 7. Runtime 的第二个核心：每一步都要更新状态，而不是只追加聊天记录

这是很多伪 Agent 实现最常见的问题。它们看起来在多步循环，但内部实际上只是在不断追加 messages，然后希望模型自己从长历史里维持一切。

这种方式能跑，但会越来越不稳。因为聊天记录不等于状态。

真正的状态更新通常至少应该显式记录：

- 已经读取了哪些 resource
- 已经执行了哪些 tool
- 拿到了哪些被确认的事实
- 当前仍然未解决的问题是什么
- 当前轮是否接近可终止条件

这样做的价值，不只是为了代码整洁，而是为了让 Runtime 可以不完全依赖模型自己从冗长上下文里复原任务结构。

In [ ]:
state_update = {
    "read_resources": ["project://case/jd_sample"],
    "confirmed_facts": ["岗位强调 system prompt、tool use、agent runtime 设计能力"],
    "pending_questions": ["候选人经历中是否有可证明的 MCP 实战案例"],
    "last_action": "resource_read",
}

print(state_update)

## 8. Tool Execution 不是调用成功就完事，而是执行结果要变成下一轮推理条件

一个常见误区是把 Runtime 写成“模型给工具意图 -> 系统执行 -> 把结果原样塞回去”。形式上没错，但如果只做到这一步，系统很容易在工具结果上继续失控。

更成熟的处理方式通常是：

- 先校验工具意图是否合法
- 再执行工具
- 对结果做必要压缩、格式化或错误标注
- 再把结果以清晰语义地位写回上下文

重点不是把所有东西都自动美化，而是确保模型下一轮面对的不是噪音，而是可继续推理的外部反馈。否则，工具调用的价值会被结果污染直接抵消。

## 9. Termination Logic：什么时候停，比怎么开始更容易被低估

在一个多步 Agent Runtime 里，最危险的通常不是第一步，而是第六步、第七步、第八步系统还在无意义地继续。尤其当底层模型是本地运行时，多走的每一步都会直接转化成延迟和资源浪费。

因此，一个像样的 Runtime 必须显式判断：

- 当前答案是否已经足够满足目标
- 当前状态是否仍有关键空洞
- 连续多步是否没有获得任何新信息
- 是否已经达到最大步数或失败阈值

终止逻辑不是一个后处理器，而是 Runtime 是否真正像系统的核心标志之一。因为一个不知道何时收束的 Agent，并不是更聪明，而只是更失控。

## 10. Error Recovery：Runtime 的成熟度，体现在失败路径上

如果说成功路径体现的是设计意图，那么失败路径体现的就是工程成熟度。一个只在理想输入下看起来顺畅的 Runtime，往往经不起真实任务压力。

对这套项目来说，至少有几类失败路径必须被 Runtime 理解：

- 模型输出结构不合法
- 模型试图调用不存在的能力
- tool 参数缺失或语义不完整
- resource 读取失败或返回空内容
- 连续多轮结果没有推进任务

一个成熟的 Runtime 不需要在第一版就覆盖所有恢复策略，但至少要能把这些失败区分出来，并给出不同响应，而不是一律崩掉或一律重试。失败分型，往往比单次成功更能说明系统是否被真正理解。

## 11. Trace 与 Observability：Runtime 不可黑箱化

既然这套项目强调系统理解，那么 Runtime 本身就不能成为新的黑箱。它至少应该让人看见：

- 当前任务目标是什么
- 每一步模型返回了什么意图
- Runtime 把这个意图解释成了什么动作
- 调用了哪个 tool 或读取了哪个 resource
- 返回结果如何改变了状态
- 为什么在某一步结束了循环

这套 trace 不是为了日志好看，而是为了让系统具备可解释性、可调试性和可评估性。尤其在本地模型环境里，很多结构不稳定的问题只有放在完整 trace 里才看得清。

In [ ]:
runtime_trace = [
    {"step": 1, "intent": "read_resource", "target": "project://case/jd_sample"},
    {"step": 2, "intent": "read_resource", "target": "project://case/candidate_profile"},
    {"step": 3, "intent": "call_tool", "target": "extract_key_requirements"},
    {"step": 4, "intent": "call_tool", "target": "score_candidate_fit"},
    {"step": 5, "intent": "final_answer", "target": "structured_match_analysis"},
]

for item in runtime_trace:
    print(item)

## 12. 一个最小但像样的 Runtime 应该长什么样

结合前面所有讨论，一个最小但像样的 Agent Runtime 不需要很复杂，但至少应该具备这些结构：

- 任务入口与初始状态构建
- system prompt / prompt template 选择
- 模型调用层
- 意图解释层
- tool/resource 执行与读取层
- 状态更新层
- 终止与错误处理层
- trace 记录层

这八层不一定非要拆成八个模块，但如果其中大半都不存在，系统就更像一段能跑通 happy path 的脚本，而不是一个可分析、可扩展的 Runtime。

## 13. 为什么说这一章是整套项目的真正核心

如果要从整套 notebook 里挑一个最能体现工程判断力的部分，Runtime 几乎一定是最核心的一章。原因很简单：

- 这里能看出你是否真的把模型、工具、资源、prompt、MCP 当成一个系统来理解
- 这里能看出你是把 Agent 当成营销词，还是当成运行时结构
- 这里能看出你知不知道“会调模型”和“会搭系统”之间的差别

也正因为如此，这一章不应该写成 API 教程。真正要让人看到的，是你如何把前面所有概念压成一个能够运行、能够解释、能够约束的任务闭环。

## 14. Runtime 一旦成形，下一步就该进入完整案例

到这里，系统的骨架已经基本完整了：

- 模型有了
- 控制面有了
- tool calling 的机制有了
- Agent 的结构和 MCP 的能力层有了
- 本地 server 有了
- Runtime 的编排逻辑也有了

下一步最自然的事情，就是把这套系统放进一个足够具体、足够业务化的案例里，让读者看到它不是在概念上闭环，而是在任务上闭环。也就是说，下一章不该再单独讲机制，而应该开始做面向招聘展示的完整演示。

## 15. 本章结论

这一章最值得保留的判断有这些：

- 真正让系统像 Agent 的，不是模型本身，而是 Runtime 把推理变成任务闭环。
- Runtime 的核心职责不是生成文本，而是维护目标、解释意图、调度能力、更新状态和决定终止。
- resource loop 和 tool loop 都应被视为一等运行路径。
- 聊天记录不等于状态，成功调用不等于任务推进。
- 失败分型、终止逻辑和 trace 可观测性，是 Runtime 成熟度的重要标志。

下一章会把这一切放进一个完整案例里，不再只讨论结构，而是直接展示这套 Runtime 如何在 HR 能理解的业务语境里完成一次完整任务。